# Домашнее задание по теме «Деревья решений: регрессия и классификация»

В работе две части: регрессия на Wine Quality и классификация на Adult. Ты сравнишь настройки sklearn и допишешь учебные реализации деревьев.

**Максимальная оценка: 10 баллов.** Задачи 1–5 дают 1, 2, 2, 2 и 2 балла, теоретические вопросы ещё 1 балл. Баллы подзадач входят в баллы задания и не прибавляются к ним повторно.

## Часть I. Деревья регрессии

Импортируем библиотеки и [загрузим датасет Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality).

Датасет Wine Quality содержит данные о португальских винах Vinho Verde. В датасете два отдельных файла: около 1 600 записей для красного вина и почти 4 900 для белого. На каждый образец приходится 11 физических и химических характеристик (фиксированная кислотность, летучая кислотность, цитрат, остаточный сахар, хлориды, свободный и общий диоксид серы, плотность, pH, сульфаты и содержание алкоголя) и целевая метка `quality` — оценка дегустатора от 0 до 10 (на практике в диапазоне ~ 3–9).

Обычно Wine Quality используют для задач регрессии, чтобы предсказать точечную оценку. Или для бинарной классификации, например, чтобы определить «низкое» или «высокое» качество, где порог — это 6. Используем датасет для задачи регрессии и  предскажем переменную `quality`.



In [ ]:
# Импорт библиотек
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
import seaborn as sns

Теперь загрузим датасет и проведём разведочный анализ данных (EDA) и препроцессинг.

In [ ]:
# 1. Ссылки на CSV
red_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
white_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

# 2. Считываем в DataFrame, указывая разделитель ';'
df_red = pd.read_csv(red_url, sep=';')
df_white = pd.read_csv(white_url, sep=';')

# 3. Добавляем колонку с типом вина
df_red["type"] = "red"
df_white["type"] = "white"

# 4. Объединяем
df = pd.concat([df_red, df_white], ignore_index=True)

# Удаляем точные повторы до разбиения, чтобы они не попадали в разные части.
df = df.drop_duplicates().reset_index(drop=True)
df_dev, df_holdout = train_test_split(df, test_size=0.2, random_state=0)
df, df_val = train_test_split(df_dev, test_size=0.25, random_state=0)
df, df_val, df_holdout = df.copy(), df_val.copy(), df_holdout.copy()

# EDA проводим по train.
# 5. Проверка
print(df.shape)   # Сколько записей и признаков
df.head()

In [ ]:
# Нет пропусков, один категориальный признак
df.info()

In [ ]:
# Есть выбросы
df.describe().T

In [ ]:
# Много выбросов
num_cols = df.select_dtypes(include="number").columns
for col in num_cols:
    plt.figure()
    plt.boxplot(df[col])
    plt.title(col)
    plt.ylabel(col)
    plt.show()

In [ ]:
Q1, Q3 = df[num_cols].quantile(0.25), df[num_cols].quantile(0.75)
IQR = Q3 - Q1
mask = (df[num_cols] < (Q1 - 1.5*IQR)) | (df[num_cols] > (Q3 + 1.5*IQR))

print("Количество выбросов в каждом столбце")
print(mask.sum().sort_values(ascending=False))

Правило 1,5 IQR отмечает редкие значения, но не доказывает ошибку измерения. Без сведений об ошибках оставим наблюдения и исходные оценки quality. Склонность дерева к переобучению будем контролировать параметрами модели.

In [ ]:
# Сохраняем исходные значения признаков и quality.
df[num_cols].describe().T

In [ ]:
plt.hist(df['quality'], bins=np.arange(2.5, 10.5, 1))
plt.xlabel('quality')
plt.ylabel('Число объектов train')
plt.show()

In [ ]:
# 1. Считаем матрицу корреляций
corr = df.corr(numeric_only=True)

# 2. Берём только столбец quality, убираем строку quality и сортируем
corr_q = corr["quality"].drop("quality").sort_values(ascending=False)

# 3. Рисуем теплокарту
plt.figure(figsize=(6, len(corr_q) * 0.4))
sns.heatmap(corr_q.to_frame(), annot=True, cmap="vlag", cbar=False)
plt.title("Корреляция признаков с quality")
plt.ylabel("Признак")
plt.xlabel("Correlation")
plt.show()

In [ ]:
# Два заранее известных типа вина кодируем одинаково во всех частях.
for frame in [df, df_val, df_holdout]:
    frame['type_white'] = (frame['type'] == 'white').astype(int)
    frame.drop(columns='type', inplace=True)
df.head()

In [ ]:
X = df.drop(["quality"], axis=1)
y = df["quality"]

In [ ]:
X_train, y_train = X, y
X_val, y_val = df_val.drop(columns='quality'), df_val['quality']
X_holdout, y_holdout = df_holdout.drop(columns='quality'), df_holdout['quality']
print('Train / validation / test:', len(X_train), len(X_val), len(X_holdout))

### Задача 1. `sklearn`: неограниченное дерево [1 балл]  

Работаем с датасетом Wine Quality. Будем предсказывать оценку качества вина.

  




#### Задача 1.1 [0,5 балла]

- Обучи неограниченное дерево `DecisionTreeRegressor(random_state=42)` и выведи $R^2$ на validation.

- Выведи глубину дерева и число листьев.

In [ ]:
# Напиши код здесь


#### Задача 1.2 [0,5 балла]

Отрисуй первые три уровня полученного дерева с помощью `plot_tree` (параметр `max_depth = 2`).

In [ ]:
# Напиши код здесь


### Задача 2. `sklearn`: ограниченное дерево [2 балла]  

Сравни неограниченное дерево с деревьями меньшей сложности на общей validation. Не используй итоговый test для выбора параметров.

#### Задача 2.1 [0,5 балла]

- Напиши цикл, который будет проходить по различным значениям `max_depth` (от 1 до 23).

- Для каждого значения глубины обучи `DecisionTreeRegressor` с заданной глубиной на обучающих данных.

- Вычисли $R^2$ только на валидационной выборке.

- Нарисуй график зависимости валидационного $R^2$ от глубины дерева.

In [ ]:
# Напиши код здесь


#### Задача 2.2 [0,5 балла]  

- Напиши аналогичный цикл для гиперпараметра `min_samples_split`. Массив значений, которые надо проверить: `splits = [10, 20, 50, 100, 150, 200, 300, 400, 500, 600, 700, 1000, 1500, 2000]`.

- Нарисуй аналогичный график.

In [ ]:
# Напиши код здесь


#### Задача 2.3 [0,5 балла]

**Вопрос.** Как можно объяснить два нарисованных графика?

Твой ответ здесь: ...

#### Задача 2.4 [0,5 балла]

Подбери гиперпараметры с Optuna, максимизируя средний R² по `KFold(n_splits=5, shuffle=True, random_state=42)` на train. Используй до 300 попыток и 600 секунд. Для воспроизводимости задай `TPESampler(seed=42)`. После поиска выведи `study.best_params` и `study.best_value`.

In [ ]:
!pip install optuna --quiet

import optuna
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error

In [ ]:
# Напиши код здесь


### Задача 3. Свой `DecisionTreeRegressor` [2 балла]  

Задача 3 состоит из трёх подзадач. В подзадаче 3.1 и 3.2 ты напишешь методы класса, а в подзадаче 3.3 вставишь эти методы в скелет класса. Этот скелет моделирует работу деревьев регрессии. Твоя цель — дописать в этот класс два метода (из подзадач 3.1 и 3.2), которые позволяют дереву расти.



In [ ]:
# Вспомогательные функции и классы
from dataclasses import dataclass

def rss(y):
    """Residual Sum of Squares."""
    y = np.asarray(y)
    mu = y.mean()
    return np.sum((y - mu) ** 2)

@dataclass
class NodeR:
    feat: int = None
    thr: float = None
    left: "NodeR" = None
    right: "NodeR" = None
    value: float = None

In [ ]:
# Меняем только представление данных. Разбиение сохраняется.
X_train = np.asarray(X_train, dtype=float)
X_val = np.asarray(X_val, dtype=float)
X_holdout = np.asarray(X_holdout, dtype=float)
y_train, y_val, y_holdout = map(np.asarray, (y_train, y_val, y_holdout))

#### Задача 3.1 [1 балл]

- Реализуй метод `_best_split(X, y, feat_idxs)`. Метод на данном шаге ищет признак и порог, минимизирующие сумму RSS в дочерних узлах.

- Используй уже написанный метод RSS, рассчитывающий сумму квадратов остатков предсказаний.

In [ ]:
# Зададим какой-то гиперпараметр min_samples_leaf
min_samples_leaf = 500

# Напиши код здесь
# Перед тем, как начать заполнять функцию, вспомни теорию построения дерева из
# лонгрида и продумай алгоритм построения дерева регрессии

# feat_idxs — номера столбцов (признаков)
# Х и у — наблюдения в текущем узле
def _best_split(X, y, feat_idxs):
    # Лучший признак по которому надо делить, лучший порог, лучшая мера неопределённости
    best_feat, best_thr, best_rss = None, None, np.inf
    n = len(y)

    # Для каждого признака выполняем
    for f in feat_idxs:
        # Сортируй по текущему признаку
        # Сортировка позволит «идти» слева направо по значениям признака и выбирать порог
        # https://numpy.org/devdocs/reference/generated/numpy.argsort.html

        # order = ...
        # x_sorted = ...
        # y_sorted = ...

        # Идём слева направо и смотрим на результативность порогов признака
        for i in range(min_samples_leaf,
                        n - min_samples_leaf + 1):
            # Напиши if, который сработает, если два разных наблюдения будут иметь
            # одно и то же значение признака. Нужно ли тогда повторно проверять
            # результативность применения одного и того же порога признака?

            # if ...
            #     ...

            left_y  = y_sorted[:i]
            right_y = y_sorted[i:]

            # Напиши, чему будет равен общий RSS двух потомственных сплитов. Нужна
            # ли здесь нормировка?
            # После общего RSS проверь, является ли этот RSS наименьшим среди всех возможных
            # сплитов, (сравни с best_rss) и, если да, обнови best_rss, best_feat и best_thr

            # totalrss = ...
            # if ...
            #     best_rss = ...
            #     best_feat = ...
            #     best_thr  = (x_sorted[i - 1] + x_sorted[i]) / 2

    return best_feat, best_thr

#### Задача 3.2 [1 балл]

Реализуй рекурсивный метод `_fit_node(X, y, depth)`, который строит всё дерево.    

In [ ]:
# Зададим гиперпараметры
max_depth = 5
min_samples_split = 500

# Напиши код здесь

# Сейчас мы используем предыдущую функцию для того, чтобы рекурсивно построить дерево
# X, y — наблюдения в текущем узле, в начале подаётся полный датасет
# depth = 0 в начале, по мере роста дерева увеличивается
def _fit_node(X, y, depth):
    # Создаем узел
    node = NodeR()

    # Напиши стоп-условия, которые остановят рост дерева и вернут значение листа
    # Подсказка: стоп-условия связаны с гиперпараметрами выше

    # if ... or ... or (np.unique(y).size == 1):
    #     node.value = ...
    #     return ...


    # Реализуй функцию _best_split()

    # feat, thr = ...

    if feat is None:     # Если сплит не найден
        node.value = y.mean()
        return node

    node.feat, node.thr = feat, thr
    mask = X[:, feat] <= thr

    # Реализуй рекурсивную функцию _fit_node, которая будет делить узел на два
    # подузла и для каждого подузла реализовывать тот же функционал

    # node.left  = ...
    # node.right = ...
    return node

#### Задача 3.3 [Обязательная]

Перенеси функции из 3.1 и 3.2 в методы `MyDecisionTreeRegressor`. Добавь `self` и используй `self.min_samples_leaf`, `self.min_samples_split`, `self.max_depth`, `self.rss`.

Обучи класс на прежнем train и рассчитай R² на validation. Сравни со sklearn при одинаковых `max_depth=5`, `min_samples_split=2`, `min_samples_leaf=1`.

Сборка класса обязательна: без неё максимум за задачу 3 составляет 1 балл. Без оценки на validation максимум составляет 1,5 балла.

In [ ]:
from dataclasses import dataclass

@dataclass
class NodeR:
    feat: int = None
    thr: float = None
    left: "NodeR" = None
    right: "NodeR" = None
    value: float = None


class MyDecisionTreeRegressor:
    def __init__(self, max_depth=5, min_samples_split=2,
                 min_samples_leaf=1, random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state  # В этой реализации разбиения детерминированы
        self.root = None

    @staticmethod
    def rss(y):
        """Residual Sum of Squares."""
        mu = y.mean()
        return np.sum((y - mu) ** 2)

    # ---------- 3.1 ----------
    def _best_split(self, X, y, feat_idxs):
        # Напиши функцию здесь
        #
        #
        #

    # ---------- 3.2 ----------
    def _fit_node(self, X, y, depth):
        # Напиши функцию здесь
        #
        #
        #

    # ---- Публичные методы fit/predict ----

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).ravel()
        self.root = self._fit_node(X, y, depth=0)
        return self

    def _predict_one(self, x, node):
        if node.left is None:
            return node.value
        return self._predict_one(x, node.left if x[node.feat] <= node.thr else node.right)

    def predict(self, X):
        if self.root is None:
            raise ValueError("Сначала вызови fit")
        return np.array([self._predict_one(x, self.root) for x in np.asarray(X)])

In [ ]:
# Напиши функцию здесь
# Создай объект класса, обучи его и протестируй
#

Сравни R² учебной и библиотечной моделей на одинаковом разбиении. Близкие метрики сами по себе не доказывают совпадения деревьев. При равноценных разбиениях реализации могут выбрать разные признаки. Для проверки алгоритма нужны также небольшие примеры с известным оптимальным порогом. После этого однократно оцени выбранное Optuna дерево на test, не меняя настройки по результату.

## Часть II. Деревья классификации

Для второй части домашнего задания тебе понадобится [датасет Adult](https://archive.ics.uci.edu/dataset/2/adult) (`adult`).

Датасет Adult (он же Census Income) — классический набор данных, собранный во время переписи населения США 1994 года. В нём около 48 000 записей и 14 признаков: возраст, род занятий, уровень образования, семейный статус, профессия, родственники в семье, раса, пол, доходы от капитала, убытки от капитала, количество рабочих часов в неделю и страна рождения. Целевая метка — бинарная: получает человек более 50 000 $ в год или нет.

Чаще всего Adult используют для обучения и сравнения бинарных классификаторов (логистическая регрессия, деревья решений, бустинг), а ещё как тест на обработку категориальных признаков (one-hot, целевое кодирование) и работу с пропущенными значениями (в датасете некоторые поля обозначены `?`). Класс `>50K` составляет около четверти выборки, что даёт умеренный дисбаланс и повод поэкспериментировать с метриками (precision/recall, ROC-AUC).




In [ ]:
# Импорт библиотек
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
import seaborn as sns

In [ ]:
# 1. Список имён колонок из adult.names
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
]

# 2. URL-адреса
train_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
test_url  = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

# 3. Скачиваем и читаем
df_train = pd.read_csv(
    train_url,
    names=columns,
    sep=',',
    na_values='?',
    skipinitialspace=True
)
df_test = pd.read_csv(
    test_url,
    names=columns,
    sep=',',
    na_values='?',
    skipinitialspace=True,
    skiprows=1  # Пропустить строку заголовка в файле .test
)

# В учебном эксперименте используем собственное разбиение объединённых данных.
# Это не оценка на официальном test-разбиении Adult.
df = pd.concat([df_train, df_test], ignore_index=True)
df['income'] = df['income'].str.strip().str.rstrip('.')
df = df.drop_duplicates().reset_index(drop=True)
df_dev, df_holdout = train_test_split(df, test_size=0.2, random_state=42, stratify=df['income'])
df, df_val = train_test_split(df_dev, test_size=0.25, random_state=42, stratify=df_dev['income'])
df, df_val, df_holdout = df.copy(), df_val.copy(), df_holdout.copy()

In [ ]:
num_cols = df.select_dtypes('number').columns
cat_cols = df.select_dtypes('object').columns.drop('income')

In [ ]:
print(df.shape)
df.head()

Начнём с изучения целевой переменной.

In [ ]:
df['income']

In [ ]:
df['income'] = df['income'].str.strip().str.rstrip('.')

In [ ]:
print(df['income'].value_counts(normalize=True))
df['income'].value_counts().plot(kind='bar')

В дальнейшем будем учитывать, что в датасете гораздо больше людей с доходом `<=50K ` (то есть классы несбалансированы).


Теперь посмотрим на пропуски.

In [ ]:
for frame in [df, df_val, df_holdout]:
    frame.replace('?', np.nan, inplace=True)

In [ ]:
df.info()

In [ ]:
for frame in [df, df_val, df_holdout]:
    frame[cat_cols] = frame[cat_cols].fillna('Unknown')

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
# Есть ли опечатки в названиях категорий? Редкие категории?
for col in cat_cols:
    print('\n', col)
    display(df[col].value_counts().head(10))

Большое число категорий увеличивает размер one-hot представления. Объединим категории, которые занимают меньше 1% train, и применим те же правила к отложенным данным.

In [ ]:
cat_cols = df.select_dtypes('object').columns.drop('income')
for col in cat_cols:
    counts = df[col].str.strip().value_counts()
    frequent = counts[counts >= 0.01*len(df)].index
    for frame in [df, df_val, df_holdout]:
        values = frame[col].str.strip()
        frame[col] = values.where(values.isin(frequent), 'other')

In [ ]:
cat_cols = df.select_dtypes('object').columns.drop('income')
for col in cat_cols:
    print('\n', col)
    display(df[col].value_counts().head(10))

In [ ]:
# Распределения числовых переменных
for col in num_cols:
    plt.figure(figsize=(4,1.2))
    plt.boxplot(df[col].dropna(), vert=False)
    plt.title(col); plt.show()

In [ ]:
(df['capital_gain'] > 0).astype(int).value_counts()

In [ ]:
def add_features(frame):
    # age: разбиение на отрезки
    frame['age_bin'] = pd.cut(frame['age'], bins=[16,25,35,45,55,65,100],
                           labels=['17-25','26-35','36-45','46-55','56-65','66+'])

    # В этой модели исключаем вес переписного наблюдения fnlwgt.
    frame = frame.drop('fnlwgt', axis=1)

    # capital_gain/capital_loss → флаги
    frame['gain_flag'] = (frame['capital_gain'] > 0).astype(int)
    frame['loss_flag'] = (frame['capital_loss'] > 0).astype(int)

    # hours_per_week: разбиение на отрезки
    frame['hours_bin'] = pd.cut(frame['hours_per_week'],
                             bins=[0,40,55,168],
                             labels=['<=40','41-55','56+'])
    return frame

df, df_val, df_holdout = [add_features(frame) for frame in [df, df_val, df_holdout]]

In [ ]:
# Проверим результат
df[['age','age_bin', 'capital_gain','gain_flag','capital_loss','loss_flag',
          'hours_per_week','hours_bin']].head()

Выделим таргет отдельно от признаков. One-hot столбцы определим по train. Validation и test приведём к тем же столбцам, а отсутствующие заполним нулями.

In [ ]:
y_train = (df['income'] == '>50K').astype(int)
y_val = (df_val['income'] == '>50K').astype(int)
y_holdout = (df_holdout['income'] == '>50K').astype(int)
X_train = pd.get_dummies(df.drop(columns='income'), dtype=int)
X_val = pd.get_dummies(df_val.drop(columns='income'), dtype=int).reindex(columns=X_train.columns, fill_value=0)
X_holdout = pd.get_dummies(df_holdout.drop(columns='income'), dtype=int).reindex(columns=X_train.columns, fill_value=0)
X_train.head()

In [ ]:
X_train.info(verbose=False)

In [ ]:
print("Число признаков:", X_train.shape[1])

In [ ]:
print('Train / validation / test:', len(X_train), len(X_val), len(X_holdout))

### Задача 4. `sklearn`: ограниченное дерево [2 балла]

Применим post-pruning. Сначала растёт полное дерево, затем из него удаляются ветви с учётом штрафа за число листьев. Параметр `ccp_alpha` управляет величиной этого штрафа. Последовательность допустимых значений возвращает `cost_complexity_pruning_path(X_train, y_train)`.

#### Задача 4.1 [1 балл]

Для критериев Джини и энтропии получи путь обрезки дерева на train. В этом эксперименте используй `class_weight='balanced'` и `random_state=42` во всех сравниваемых моделях.

Для каждого критерия возьми не более 300 равномерно расположенных индексов массива уникальных неотрицательных `ccp_alphas`. Если значений меньше 300, используй все. Для каждого alpha обучи дерево и рассчитай ROC-AUC по `predict_proba(X_val)[:, 1]`.

Сохрани `val_gini`, `val_entropy` той же длины, что соответствующие массивы alpha. Accuracy при дисбалансе может скрывать ошибки редкого класса. Здесь сравниваем ранжирование по ROC-AUC, а затем рассмотрим ошибки классов в confusion matrix. Сам дисбаланс не делает accuracy бессмысленной.

In [ ]:
from sklearn.metrics import roc_auc_score
# Напиши код здесь


#### Задача 4.2 [1 балл]

Построй validation ROC-AUC для всех проверенных alpha, включая значения с низким качеством. Чтобы различать малые alpha и сохранить alpha=0, используй `plt.xscale("symlog", linthresh=1e-5)`. Для каждого критерия выбери alpha по validation и построй confusion matrix на validation. Затем выбери один из двух вариантов по validation ROC-AUC, при равенстве оставь Gini. Однократно оцени его на test. По test больше не меняй alpha или критерий.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Напиши код здесь


### Задача 5. Расширение `MyDecisionTreeClassifier` [2 балла]

В учебный класс добавим порог локального уменьшения неоднородности `min_gain` и веса классов. Этот порог останавливает рост дерева во время обучения и отличается от post-pruning через `ccp_alpha` в sklearn. Здесь `min_gain` относится к текущему узлу. Параметр `min_impurity_decrease` в sklearn дополнительно взвешивает уменьшение долей узла во всей обучающей выборке.

In [ ]:
from typing import Optional, Union, Dict

class _Node:
    """Вершина дерева (и внутренний узел, и лист)."""
    def __init__(self, feat=None, thr=None, left=None, right=None,
                 proba=None, label=None):
        self.feat = feat      # Индекс признака для сплита
        self.thr = thr        # Порог сплита
        self.left = left      # Левая ветка
        self.right = right    # Правая ветка
        self.proba = proba    # Вероятности классов (только в листе)
        self.label = label    # Метка класса (только в листе)

    def is_leaf(self):
        """Возвращает True, если вершина является листом."""
        return self.label is not None


class MyDecisionTreeClassifier:
    """
    Мини-реализация дерева классификации (критерий Gini,
    бинарные сплиты по порогу).
    """

    def __init__(
        self,
        max_depth: Optional[int] = None,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
        criterion: str = "gini",                     # #### NEW
        class_weight: Union[str, Dict[int, float], None] = None,  # #### NEW — твоё следующее задание
        min_gain: float = 0.0,                      # #### NEW — твоё следующее задание
        random_state: Optional[int] = None           # #### NEW
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion                   # #### NEW
        self.class_weight = class_weight             # #### NEW — твоё следующее задание
        self.min_gain = float(min_gain)            # #### NEW — твоё следующее задание
        self.random_state = random_state  # Разбиения перебираются детерминированно

    # ---------- Методы, реализованные вручную ----------
    def _make_leaf(self, y_subset: np.ndarray) -> _Node:
        """
        Превращает подмножество объектов в лист и
        возвращает объект `_Node` с полями `proba`, `label`.
        """
        counts = np.bincount(y_subset, weights=self.class_weights_[y_subset], minlength=self.n_classes_)
        proba  = counts / counts.sum()
        return _Node(proba=proba, label=int(proba.argmax()))

    def _predict_one(self, x_row: np.ndarray, node: _Node) -> int:
        """
        Спускает один объект по дереву до листа
        и возвращает предсказанный класс.
        """
        while not node.is_leaf():
            node = node.left if x_row[node.feat] <= node.thr else node.right
        return node.label

    def predict(self, X) -> np.ndarray:
        """
        Возвращает вектор прогнозов для всей матрицы `X`.

        Обрати внимание: мы сразу преобразуем X в numpy-массив,
        чтобы не индексировать строки как строки или pandas-объекты.
        """
        X_arr = np.asarray(X, dtype=float)
        return self.classes_[np.array([self._predict_one(row, self.root_) for row in X_arr])]

    # ----------------------------------------------------

    # ---------- Готовые вспомогательные методы ----------
    @staticmethod
    def _gini(counts: np.ndarray) -> float:
        p = counts / counts.sum()
        return 1.0 - (p ** 2).sum()

    @staticmethod
    def _entropy(counts: np.ndarray) -> float:             # NEW
        p = counts / counts.sum()
        p = p[p > 0]
        return -(p * np.log2(p)).sum()

    def _impurity(self, counts: np.ndarray) -> float:      # NEW
        return self._gini(counts) if self.criterion == "gini" else self._entropy(counts)

    def _best_split(self, X: np.ndarray, y: np.ndarray):
        """
        Перебирает все признаки и ищет порог,
        дающий максимальный прирост информации.
        Возвращает (feat, thr). Если сплита нет — (None, None).
        """
        best_gain, best_feat, best_thr = -np.inf, None, None
        parent_imp = self._impurity(np.bincount(y, weights=self.class_weights_[y], minlength=self.n_classes_))
        for feat in range(X.shape[1]):
            xs = X[:, feat]
            order = np.argsort(xs)
            xs_sort, y_sort = xs[order], y[order]
            right = np.bincount(y_sort, weights=self.class_weights_[y_sort], minlength=self.n_classes_).astype(float)
            left  = np.zeros_like(right)
            for i in range(1, len(y_sort)):
                cls = y_sort[i - 1]
                left[cls]  += self.class_weights_[cls]
                right[cls] -= self.class_weights_[cls]
                if xs_sort[i] == xs_sort[i - 1]:
                    continue
                if i < self.min_samples_leaf or len(y_sort) - i < self.min_samples_leaf:
                    continue
                wL = left.sum() / (left.sum() + right.sum())
                wR = 1 - wL
                gain = parent_imp - (wL * self._impurity(left) + wR * self._impurity(right))
                if gain > best_gain:
                    best_gain, best_feat = gain, feat
                    best_thr = (xs_sort[i] + xs_sort[i - 1]) / 2
        return best_feat, best_thr

    def _grow(self, X: np.ndarray, y: np.ndarray, depth: int = 0) -> _Node:
        """
        Рекурсивно строит дерево, возвращая корневой узел `_Node`.
        """
        if (self.max_depth is not None and depth == self.max_depth) or \
           len(np.unique(y)) == 1 or len(y) < self.min_samples_split:
            return self._make_leaf(y)

        feat, thr = self._best_split(X, y)
        if feat is None:
            return self._make_leaf(y)

        mask = X[:, feat] <= thr
        left  = self._grow(X[mask],  y[mask],  depth + 1)
        right = self._grow(X[~mask], y[~mask], depth + 1)
        return _Node(feat, thr, left, right)

    def fit(self, X, y):
        """
        Строит дерево по обучающим данным.
        Если `y` содержит строковые метки, сначала преобразует их в целые.
        """
        # Приводим X, y к numpy-массиву целых
        X_arr = np.asarray(X, dtype=float)
        y_arr = np.asarray(y)

        self.classes_, y_arr = np.unique(y_arr.ravel(), return_inverse=True)
        self.n_classes_ = len(self.classes_)
        self.class_weights_ = np.ones(self.n_classes_)

        self.root_ = self._grow(X_arr, y_arr)
        return self

Давай дополним этот класс.





#### Задача 5.1 [0,5 балла]

В `_best_split` используй выбранный `criterion` в родителе и обоих потомках. После поиска лучшего сплита верни `(None, None)`, если уменьшение взвешенной неоднородности не превосходит `self.min_gain`. Уменьшение здесь считается относительно текущего узла. При пороге 0 сохраняй только строго улучшающие разбиения.

In [ ]:
# Напиши код здесь
def _best_split(self, X: np.ndarray, y: np.ndarray):
    pass

#### Задача 5.2 [0,5 балла]

В `fit` закодируй все метки через `np.unique(..., return_inverse=True)` и сохрани `self.classes_`. Вычисли `self.class_weights_`: единицы без весов, n/(K*n_k) для balanced или значения словаря по исходным меткам. Готовые методы базового класса уже используют этот массив в листьях и при выборе сплита. Проверь, что веса действительно меняют результат на небольшом несбалансированном примере.

In [ ]:
def fit(self, X, y):
    X_arr = np.asarray(X, dtype=float)
    self.classes_, y_arr = np.unique(np.asarray(y).ravel(), return_inverse=True)
    self.n_classes_ = len(self.classes_)
    # Создай self.class_weights_ по правилу из условия.
    # Напиши код здесь
    self.root_ = self._grow(X_arr, y_arr)
    return self

#### Задача 5.3 [Обязательная]

Собери итоговый класс. В нём веса объектов передаются явно через аргумент `w`. Адаптируй `_best_split` к взвешенным частотам. В `fit` создай веса объектов как `self.class_weights_[y_arr]` и передай их в `_grow`. `min_samples_leaf` и `min_samples_split` ограничивают число объектов, а не сумму весов.

Без собранного работоспособного класса баллы за задачу 5 не начисляются.

In [ ]:
from typing import Optional, Union, Dict


class _Node:
    def __init__(self, feat=None, thr=None, left=None, right=None, proba=None, label=None):
        self.feat = feat
        self.thr = thr
        self.left = left
        self.right = right
        self.proba = proba
        self.label = label

    def is_leaf(self):
        return self.label is not None


class MyDecisionTreeClassifier:
    def __init__(
        self,
        max_depth: Optional[int] = None,
        min_samples_split: int = 2,
        min_samples_leaf: int = 1,
        criterion: str = "gini",
        class_weight: Union[str, Dict[int, float], None] = None,
        min_gain: float = 0.0,
        random_state: Optional[int] = None
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.class_weight = class_weight
        self.min_gain = float(min_gain)
        self.random_state = random_state  # Детерминированный перебор разбиений

    @staticmethod
    def _gini(counts: np.ndarray) -> float:
        p = counts / counts.sum()
        return 1.0 - (p ** 2).sum()

    @staticmethod
    def _entropy(counts: np.ndarray) -> float:
        p = counts / counts.sum()
        p = p[p > 0]
        return -(p * np.log2(p)).sum()

    def _impurity(self, counts: np.ndarray) -> float:
        return self._gini(counts) if self.criterion == "gini" else self._entropy(counts)

    def _make_leaf(self, y_subset: np.ndarray, w_subset: np.ndarray) -> _Node:
        counts = np.bincount(y_subset, weights=w_subset, minlength=self.n_classes_)
        proba = counts / counts.sum()
        return _Node(proba=proba, label=int(proba.argmax()))

    def _best_split(self, X: np.ndarray, y: np.ndarray, w: np.ndarray):

        # Напиши свой метод здесь

        pass

    def _grow(self, X: np.ndarray, y: np.ndarray, w: np.ndarray, depth: int = 0) -> _Node:
        if (self.max_depth is not None and depth == self.max_depth) or \
           len(np.unique(y)) == 1 or len(y) < self.min_samples_split:
            return self._make_leaf(y, w)
        feat, thr = self._best_split(X, y, w)
        if feat is None:
            return self._make_leaf(y, w)
        mask = X[:, feat] <= thr
        left = self._grow(X[mask], y[mask], w[mask], depth + 1)
        right = self._grow(X[~mask], y[~mask], w[~mask], depth + 1)
        return _Node(feat, thr, left, right)

    def _predict_one(self, x_row: np.ndarray, node: _Node) -> int:
        while not node.is_leaf():
            node = node.left if x_row[node.feat] <= node.thr else node.right
        return node.label

    def predict(self, X) -> np.ndarray:
        X_arr = np.asarray(X, dtype=float)
        if not hasattr(self, "root_"):
            raise ValueError("Сначала вызови fit")
        return self.classes_[np.array([self._predict_one(row, self.root_) for row in X_arr])]

    def predict_proba(self, X):
        X_arr = np.asarray(X, dtype=float)
        if not hasattr(self, "root_"):
            raise ValueError("Сначала вызови fit")
        probs = []
        for row in X_arr:
            node = self.root_
            while not node.is_leaf():
                node = node.left if row[node.feat] <= node.thr else node.right
            probs.append(node.proba)
        return np.vstack(probs)

    def fit(self, X, y):
        X_arr = np.asarray(X, dtype=float)
        self.classes_, y_arr = np.unique(np.asarray(y).ravel(), return_inverse=True)
        self.n_classes_ = len(self.classes_)
        # Создай self.class_weights_ по решению 5.2.
        # Напиши код здесь
        sample_w = self.class_weights_[y_arr]
        self.root_ = self._grow(X_arr, y_arr, sample_w)
        return self


#### Задача 5.4 [1 балл]

Обучи итоговый класс на Adult с `criterion="gini"`. Начни с `max_depth=5`, `min_samples_leaf=20`, `class_weight="balanced"`, `min_gain=0`. Рассчитай validation ROC-AUC по вероятности класса 1 и сравни со sklearn при тех же ограничениях и весах. Если меняешь настройки, используй validation. Фиксированного обязательного порога ROC-AUC здесь нет.

In [ ]:
# Напиши код здесь: создай custom_tree, обучи на train и рассчитай validation ROC-AUC.
# Для столбца вероятности используй np.flatnonzero(custom_tree.classes_ == 1)[0].

### Теоретические вопросы [1 балл]

1. Как влияют гиперпараметры `max_depth, min_samples_split, min_samples_leaf` на bias-variance дерева решений? [0.5 балла]

2. Почему один и тот же датасет может давать такие разные деревья при малых изменениях данных? [0.5 балла]


----------------------------------------------------------------